# DATA Analysis - VANGUARD A/B TEST
---



**Completion Rate**: The proportion of users who reach the final 'confirm' step.
**Time Spent on Each Step**: The average duration users spend on each step.
**Error Rates**: If there's a step where users go back to a previous step, it may indicate confusion or an error. You should consider moving from a later step to an earlier one as an error.

IMPORT LIBRARIES

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_style("whitegrid")


1. LOAD THE DATASETS 

In [4]:
data_path = "data/processed"
df_exde = pd.read_csv(f"{data_path}/df_exde.csv")
df_web_3 = pd.read_csv(f"{data_path}/df_web_3.csv")

## Phase 2 — Performance Metrics

DATASET: Digital Footprints

In [5]:
df_web_3.head()

,client_id,visitor_id,visit_id,process_step,date_time,next_time,sec_spent
0,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,2017-04-12 20:19:45,9.0
1,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,2017-04-12 20:20:31,46.0
2,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,2017-04-12 20:22:05,94.0
3,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,2017-04-12 20:23:09,64.0
4,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,done,0.0


In [8]:
df_web_3.groupby('process_step')['sec_spent'].mean().round(2)

process_step
confirm     33.99
start       48.47
step_1      55.33
step_2      88.28
step_3     122.09
Name: sec_spent, dtype: float64

In [10]:
df_web_3.groupby(['client_id', 'process_step'])['sec_spent'].sum()

client_id  process_step
169        confirm           0.0
           start             9.0
           step_1           46.0
           step_2           94.0
           step_3           64.0
                           ...  
9999875    confirm           0.0
           start             7.0
           step_1           99.0
           step_2          191.0
           step_3          221.0
Name: sec_spent, Length: 493122, dtype: float64

DATASET: Web_2

In [164]:
df_web_2.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


In [165]:
df_web_2.shape

(412264, 5)

In [166]:
df_web_2["client_id"].nunique()

67430

In [167]:
# merging both databases

df_web_3 = pd.concat([df_web_1,df_web_2], axis = 0, ignore_index = True)
df_web_3

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04
...,...,...,...,...,...
755400,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:46:10
755401,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:45:29
755402,9668240,388766751_9038881013,922267647_3096648104_968866,step_1,2017-05-24 18:44:51
755403,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:44:34


In [168]:
df_web_3.dtypes

client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
dtype: object

**Sorting Date_time Column**

In [169]:
df_web_3['date_time'] = pd.to_datetime(df_web_3['date_time'])

In [170]:
df_web_3 = df_web_3.sort_values(by=['client_id', 'visit_id', 'date_time'])

In [171]:
df_web_3.head()

,client_id,visitor_id,visit_id,process_step,date_time
285515,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36
285514,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45
285513,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31
285512,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05
285511,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09


In [172]:
df_web_3['next_time'] = df_web_3.groupby(['client_id', 'visit_id'])['date_time'].shift(-1)

In [173]:
df_web_3['sec_spent'] = (df_web_3['next_time'] - df_web_3['date_time']).dt.total_seconds()

In [174]:
df_web_3.head()

,client_id,visitor_id,visit_id,process_step,date_time,next_time,sec_spent
285515,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,2017-04-12 20:19:45,9.0
285514,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,2017-04-12 20:20:31,46.0
285513,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,2017-04-12 20:22:05,94.0
285512,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,2017-04-12 20:23:09,64.0
285511,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,NaT,NaN


**Dealing with NULL VALUES**

In [175]:
df_web_3['sec_spent'] = df_web_3['sec_spent'].fillna(0)

In [176]:
# finding nulls
df_web_3.isna().sum()

client_id            0
visitor_id           0
visit_id             0
process_step         0
date_time            0
next_time       159112
sec_spent            0
dtype: int64

In [177]:
df_web_3['next_time'] = df_web_3['next_time'].fillna("done")

In [180]:
df_web_3.head()

,client_id,visitor_id,visit_id,process_step,date_time,next_time,sec_spent
285515,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,2017-04-12 20:19:45,9.0
285514,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,2017-04-12 20:20:31,46.0
285513,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,2017-04-12 20:22:05,94.0
285512,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,2017-04-12 20:23:09,64.0
285511,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,done,0.0


In [178]:
df_web_3.isna().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
next_time       0
sec_spent       0
dtype: int64

**Dealing with Duplicates**

In [183]:
# duplicates sum
df_web_3.duplicated().sum()

np.int64(1787)

## Descriptive Analysis Numerical Variables

In [186]:
df_web_3["client_id"].nunique()

120157

In [187]:
df_web_3["visitor_id"].nunique()
	

130236

In [ ]:
df_web_3["process_step"].nunique()

5

In [181]:
df_web_3["sec_spent"].value_counts()

sec_spent
0.0       170829
4.0        15439
5.0        15339
6.0        14188
7.0        13577
           ...  
3361.0         1
1869.0         1
5020.0         1
2867.0         1
1626.0         1
Name: count, Length: 2309, dtype: int64

In [182]:
df_web_3["sec_spent"].describe().round(2)

count    755405.00
mean         65.93
std         188.20
min           0.00
25%           3.00
50%          21.00
75%          63.00
max       40235.00
Name: sec_spent, dtype: float64

## Exporting Database

In [188]:
# exporting database
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

df_web_3.to_csv(f"{output_dir}/df_web_3.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path)
    print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (2,987,098 bytes)
  df_web_3.csv (81,420,206 bytes)
